# 9.4 推理框架 (Inference Frameworks)

推理框架将各种优化技术整合为生产级服务，是大模型部署的核心基础设施。

本节涵盖：
- TensorRT-LLM
- vLLM
- TGI
- ONNX Runtime

## 1. 推理框架对比

**TensorRT-LLM**（NVIDIA）：
- 基于TensorRT的高性能推理引擎
- 支持FP8/INT4/INT8量化、KV缓存、Continuous Batching
- 需要预编译模型（build阶段），部署后不可更改
- 适合NVIDIA GPU，性能最高

**vLLM**：
- 开源高性能推理引擎
- PagedAttention核心创新，高效KV缓存管理
- 支持Continuous Batching、Speculative Decoding
- 易于使用，支持多种模型

**TGI（Text Generation Inference）**（HuggingFace）：
- 生产级文本生成服务
- 支持Flash Attention、Paged Attention、量化
- 内置监控和API服务

**ONNX Runtime**：
- 跨平台推理引擎
- 支持CPU/GPU/DPU等多种硬件
- 适合边缘部署

In [ ]:
import torch
import time

torch.manual_seed(42)

model = torch.nn.Sequential(
    torch.nn.Linear(512, 2048),
    torch.nn.ReLU(),
    torch.nn.Linear(2048, 512)
)

x = torch.randn(1, 512)

n_warmup = 10
for _ in range(n_warmup):
    _ = model(x)

n_runs = 100
start = time.time()
for _ in range(n_runs):
    _ = model(x)
eager_time = (time.time() - start) / n_runs * 1000

model_compiled = torch.compile(model)
for _ in range(n_warmup):
    _ = model_compiled(x)

start = time.time()
for _ in range(n_runs):
    _ = model_compiled(x)
compiled_time = (time.time() - start) / n_runs * 1000

print('=== Inference Optimization Demo ===')
print(f'Model: 2-layer MLP (512->2048->512)')
print(f'Input: {x.shape}')
print(f'\nEager mode: {eager_time:.2f} ms')
print(f'torch.compile: {compiled_time:.2f} ms')
if compiled_time < eager_time:
    print(f'Speedup: {eager_time/compiled_time:.2f}x')

print(f'\n=== Production Framework Comparison ===')
frameworks = {
    'TensorRT-LLM': 'Highest perf, NVIDIA only, pre-compiled',
    'vLLM': 'Best open-source, PagedAttention, easy to use',
    'TGI': 'Production-ready, HuggingFace ecosystem',
    'ONNX Runtime': 'Cross-platform, edge deployment',
}
for name, desc in frameworks.items():
    print(f'  {name:16s}: {desc}')
print(f'\nKey: Choose framework based on deployment requirements.')
print(f'vLLM for ease of use, TensorRT-LLM for max performance.')